In [13]:
import os
import json
import glob
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, Image, clear_output

# ==========================================
# 1. CONFIGURATION
# ==========================================

BASE_PATH = "/pscratch/sd/t/tihsu/database/GridStudy_v2/method/"
JSON_GRID_PATH = "config/sample_list.json" 

# ==========================================
# 2. DATA LOADER
# ==========================================

def load_reference_grid(json_path):
    if not os.path.exists(json_path): return None, None
    with open(json_path, 'r') as f:
        config = json.load(f)
    mx_set = set()
    my_set = set()
    for mass_set in config.get("signal", []):
        match = re.search(r"MX-([\d\.]+)_MY-([\d\.]+)", mass_set)
        if match:
            # Force INT for cleaner axis
            mx_set.add(int(float(match.group(1))))
            my_set.add(int(float(match.group(2))))
    if mx_set:
        return sorted(list(mx_set)), sorted(list(my_set))
    return None, None

def parse_mass_from_string(s):
    match = re.search(r"MX-([\d\.]+)_MY-([\d\.]+)", s)
    if match:
        return int(float(match.group(1))), int(float(match.group(2)))
    return None, None

def load_grid_data(base_path):
    records = []

    # 1. Individual
    ind_search = os.path.join(base_path, "*", "individual", "*", "*.json")
    for json_path in glob.glob(ind_search):
        try:
            parts = json_path.split(os.sep)
            method = parts[-4]
            train_type = "individual"
            folder_name = parts[-2] 
            mx, my = parse_mass_from_string(folder_name)

            if mx is None: continue
            dirname = os.path.dirname(json_path)
            png_candidates = glob.glob(os.path.join(dirname, "*.png"))
            png_path = next((p for p in png_candidates if "score" in p), None)

            with open(json_path, 'r') as f: data = json.load(f)
            records.append({
                "Method": method, "Type": train_type, "ID": f"{method} ({train_type})",
                "MX": mx, "MY": my, "auc": data.get("auc", np.nan),
                "max_sic": data.get("max_sic", np.nan), "png_path": png_path,
                "time": data.get("fitting_time", np.nan)
            })
        except: continue



    # 2. Parameterized

    param_search = os.path.join(base_path, "*", "parameterized", "*", "*.json")
    for json_path in glob.glob(param_search):
        try:
            parts = json_path.split(os.sep)
            method = parts[-4]
            train_type = "parameterized"
            filename = os.path.basename(json_path)
            mx, my = parse_mass_from_string(filename)
            if mx is None: continue
            dirname = os.path.dirname(json_path)
            png_matches = glob.glob(os.path.join(dirname, f"*score*MX-{mx}*MY-{my}*.png"))
            if not png_matches:
                 png_matches = glob.glob(os.path.join(dirname, f"*score*MX-{float(mx)}*MY-{float(my)}*.png"))
            png_path = png_matches[0] if png_matches else None
            with open(json_path, 'r') as f: data = json.load(f)
            records.append({
                "Method": method, "Type": train_type, "ID": f"{method} ({train_type})",
                "MX": mx, "MY": my, "auc": data.get("auc", np.nan),
                "max_sic": data.get("max_sic", np.nan), "png_path": png_path,
                "time": data.get("fitting_time", np.nan)
            })
        except: continue
    return pd.DataFrame(records)


# ==========================================
# 3. DASHBOARD CLASS (REVISED)
# ==========================================

class GridDashboard:
    def __init__(self):
        self.df = pd.DataFrame()
        self.expected_mx = []
        self.expected_my = []
        
        # --- Widgets ---
        self.btn_load = widgets.Button(description="Refresh", button_style='primary', icon='refresh', layout=widgets.Layout(width='100px'))
        self.dd_metric = widgets.Dropdown(description="Metric:", options=['auc', 'max_sic', "time"], value='auc', layout=widgets.Layout(width='200px'))
        
        # Tuning Widgets
        self.cb_show_text = widgets.Checkbox(value=True, description='Values')
        self.slider_font = widgets.IntSlider(value=10, min=6, max=18, step=1, description='Font:', readout=True, layout=widgets.Layout(width='200px'))
        self.slider_prec = widgets.IntSlider(value=3, min=0, max=5, step=1, description='Decimals:', readout=True, layout=widgets.Layout(width='200px'))
        
        self.dd_baseline = widgets.Dropdown(description="Baseline:", layout=widgets.Layout(width='300px'))
        
        self.dd_mx = widgets.Dropdown(description="Select MX:")
        self.dd_my = widgets.Dropdown(description="Select MY:")
        
        self.out_status = widgets.Output()
        self.out_heatmap = widgets.Output()
        self.out_details = widgets.Output()
        
        # --- Bindings ---
        self.btn_load.on_click(self.load_data)
        
        self.dd_metric.observe(self.render_plots, names='value')
        self.dd_baseline.observe(self.render_plots, names='value')
        self.cb_show_text.observe(self.render_plots, names='value')
        self.slider_font.observe(self.render_plots, names='value')
        self.slider_prec.observe(self.render_plots, names='value')
        
        self.dd_mx.observe(self.update_my_options, names='value')
        self.dd_my.observe(self.render_details, names='value')

        # --- Layout ---
        ctrl_row1 = widgets.HBox([self.btn_load, self.dd_metric, self.out_status])
        ctrl_row2 = widgets.HBox([self.dd_baseline, self.cb_show_text, self.slider_font, self.slider_prec])
        
        self.ui = widgets.VBox([
            ctrl_row1,
            ctrl_row2,
            widgets.HTML("<hr>"),
            self.out_heatmap,
            widgets.HTML("<hr><h3 style='color:#333'>Drill Down Inspector</h3>"),
            widgets.HBox([self.dd_mx, self.dd_my]),
            self.out_details
        ])
        
        self.load_data(None)

    def load_data(self, b):
        with self.out_status:
            clear_output()
            print("Loading...")
            self.expected_mx, self.expected_my = load_reference_grid(JSON_GRID_PATH)
            
            df_raw = load_grid_data(BASE_PATH)
            if not df_raw.empty:
                self.df = df_raw.drop_duplicates(subset=['ID', 'MX', 'MY'], keep='last')
                if not self.expected_mx:
                    self.expected_mx = sorted(list(set(self.df['MX'].unique())))
                    self.expected_my = sorted(list(set(self.df['MY'].unique())))
                print(f"Loaded {len(self.df)} records.")
            else:
                self.df = pd.DataFrame(columns=['ID', 'MX', 'MY', 'auc', 'max_sic', "time", 'png_path'])
                print("No data found.")

        if self.df.empty and not self.expected_mx: return

        methods = sorted(self.df['ID'].unique()) if not self.df.empty else []
        self.dd_baseline.options = methods
        
        if methods:
            default = next((m for m in methods if 'base' in m.lower()), methods[0])
            self.dd_baseline.value = default
        
        self.dd_mx.options = self.expected_mx
        if self.expected_mx:
            self.dd_mx.value = self.expected_mx[0]
            self.update_my_options({'new': self.expected_mx[0]})
        
        self.render_plots(None)

    def update_my_options(self, change):
        valid_mys = self.expected_my 
        self.dd_my.options = valid_mys
        if self.dd_my.value not in valid_mys and valid_mys:
            self.dd_my.value = valid_mys[0]
        self.render_details(None)

    def get_full_pivot(self, method_id, metric):
        sub = self.df[self.df['ID'] == method_id]
        if sub.empty: 
            return pd.DataFrame(index=self.expected_my, columns=self.expected_mx)
            
        piv = sub.pivot_table(index='MY', columns='MX', values=metric, aggfunc='mean')
        piv = piv.reindex(index=self.expected_my, columns=self.expected_mx)
        return piv.sort_index(ascending=False)

    def render_plots(self, change):
        if not self.expected_mx or not self.dd_baseline.value: return
        
        metric = self.dd_metric.value
        base_id = self.dd_baseline.value
        all_methods = sorted(self.df['ID'].unique())
        
        show_text = self.cb_show_text.value
        font_size = self.slider_font.value
        decimals = self.slider_prec.value
        fmt_str = f".{decimals}f"
        
        with self.out_heatmap:
            clear_output(wait=True)
            
            n_methods = len(all_methods)
            # Taller plots
            fig, axes = plt.subplots(n_methods, 2, figsize=(16, 6 * n_methods), dpi=100)
            
            if n_methods == 1: axes = axes.reshape(1, -1)
            
            p_base = self.get_full_pivot(base_id, metric)
            g_min, g_max = self.df[metric].min(), self.df[metric].max()

            for i, method in enumerate(all_methods):
                p_curr = self.get_full_pivot(method, metric)
                
                # --- Plot 1: Absolute Metric ---
                ax_abs = axes[i, 0]
                sns.heatmap(p_curr, ax=ax_abs, annot=show_text, fmt=fmt_str, 
                            cmap="PuBu", vmin=g_min, vmax=g_max, cbar=True,
                            annot_kws={"size": font_size})
                
                title = f"{method}"
                if method == base_id: title += " (BASELINE)"
                ax_abs.set_title(title, fontsize=12, fontweight='bold')
                ax_abs.set_ylabel("MY")
                
                if i == n_methods - 1: ax_abs.set_xlabel("MX")
                else: ax_abs.set_xlabel("")

                # --- Plot 2: Ratio vs Baseline ---
                ax_ratio = axes[i, 1]
                p_ratio = p_curr / p_base
                
                # Using RdBu_r: Blue (>1) is usually "Good" for AUC. 
                # Note: For Time, Blue means >1 (Slower), which is bad, but keeping heatmap consistent is usually less confusing.
                sns.heatmap(p_ratio, ax=ax_ratio, annot=show_text, fmt=fmt_str, 
                            cmap="RdBu_r", center=1.0, vmin=0.5, vmax=1.5, cbar=True,
                            annot_kws={"size": font_size})
                
                ax_ratio.set_title(f"Ratio: {method} / Baseline")
                
                if i == n_methods - 1: ax_ratio.set_xlabel("MX")
                else: ax_ratio.set_xlabel("")
                ax_ratio.set_ylabel("")
                ax_ratio.set_yticks([]) 

            plt.tight_layout()
            plt.show()

    def render_details(self, change):
        if not self.dd_mx.value or not self.dd_my.value: return

        mx, my = self.dd_mx.value, self.dd_my.value
        all_methods = sorted(self.df['ID'].unique())
        base_id = self.dd_baseline.value
        
        with self.out_details:
            clear_output(wait=True)
            
            display(widgets.HTML(f"<h4>Data Point: MX={mx}, MY={my}</h4>"))
            
            # --- 1. Summary Table ---
            html = "<table border='1' style='width:90%; border-collapse:collapse; text-align:center; font-family:sans-serif;'>"
            html += "<tr style='background:#333; color:white;'><th>Method</th><th>AUC</th><th>Time(s)</th><th>Max SIC</th></tr>"
            
            # Get baseline values for calculation
            base_row = self.df[(self.df['ID'] == base_id) & (self.df['MX'] == mx) & (self.df['MY'] == my)]
            
            base_vals = {}
            if not base_row.empty:
                base_vals['auc'] = base_row.iloc[0]['auc']
                base_vals['time'] = base_row.iloc[0]['time']
                base_vals['max_sic'] = base_row.iloc[0]['max_sic']
            else:
                base_vals = {'auc': None, 'time': None, 'max_sic': None}

            # Helper to format cell
            def format_cell(val, base_val, higher_is_better=True):
                if pd.isna(val): return "-"
                
                base_str = f"{val:.4f}"
                if base_val is None or pd.isna(base_val) or base_val == 0:
                    return base_str
                
                ratio = val / base_val
                
                # Determine Color
                color = "black"
                is_good = False
                
                if higher_is_better:
                    if ratio > 1.0001: is_good = True
                    elif ratio < 0.9999: is_good = False
                    else: color = "#555" # Equal
                else: # Lower is better (Time)
                    if ratio < 0.9999: is_good = True
                    elif ratio > 1.0001: is_good = False
                    else: color = "#555"

                if color == "black":
                    color = "green" if is_good else "red"
                    
                return f"{base_str} <span style='color:{color}; font-weight:bold'>({ratio:.2f}x)</span>"

            for method in all_methods:
                row = self.df[(self.df['ID'] == method) & (self.df['MX'] == mx) & (self.df['MY'] == my)]
                
                bg_color = "#eafaf1" if method == base_id else "white"
                fw = "bold" if method == base_id else "normal"
                
                if row.empty:
                    html += f"<tr style='background:{bg_color}'><td>{method}</td><td colspan='3' style='color:#aaa'>No Data</td></tr>"
                    continue
                
                r = row.iloc[0]
                
                # Generate formatted strings
                s_auc = format_cell(r['auc'], base_vals['auc'], higher_is_better=True)
                s_time = format_cell(r['time'], base_vals['time'], higher_is_better=False) # Time: Lower is Green
                s_sic = format_cell(r['max_sic'], base_vals['max_sic'], higher_is_better=True)

                html += f"""
                <tr style='background:{bg_color}; font-weight:{fw}'>
                    <td style='text-align:left; padding-left:10px'>{method}</td>
                    <td>{s_auc}</td>
                    <td>{s_time}</td>
                    <td>{s_sic}</td>
                </tr>
                """
            html += "</table><br>"
            display(widgets.HTML(html))
            
            # --- 2. Images Grid ---
            img_items = []
            
            def create_img_card(method, row):
                style = "border:1px solid #ddd; padding:5px; margin:5px; border-radius:4px; text-align:center; background:white;"
                if method == base_id: style += " border: 2px solid #007bff; box-shadow: 0 0 5px rgba(0,0,0,0.2);"
                
                title_html = widgets.HTML(f"<div style='font-weight:bold; margin-bottom:5px;'>{method}</div>")
                
                if row.empty or not row.iloc[0]['png_path'] or not os.path.exists(row.iloc[0]['png_path']):
                    content = widgets.HTML("<div style='width:300px; height:200px; display:flex; align-items:center; justify-content:center; background:#f0f0f0; color:#aaa'>No Image</div>")
                else:
                    path = row.iloc[0]['png_path']
                    content = widgets.Image(value=open(path, "rb").read(), format='png', width=300)
                
                return widgets.VBox([title_html, content], layout=widgets.Layout(border='none', margin='0px'))

            for method in all_methods:
                row = self.df[(self.df['ID'] == method) & (self.df['MX'] == mx) & (self.df['MY'] == my)]
                card = create_img_card(method, row)
                img_items.append(card)

            display(widgets.Box(img_items, layout=widgets.Layout(
                display='flex',
                flex_flow='row wrap',
                align_items='flex-start',
                width='100%'
            )))

d = GridDashboard()
display(d.ui)